<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/10-partitioning-shuffle/00-partitions-cores-tasks.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 10.0 — Partitions vs cores vs tasks: the mental model

Count your slots, watch waves happen on a wall clock, and trace
where every partition count in a job actually came from.

Keep the Spark UI open at http://localhost:4040 throughout — this
module is one long UI exercise.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("10.0-partitions-cores-tasks")
    .master("local[*]")
    # AQE off so partition counts are exactly what the configs say
    # (Module 7 convention: static plans while teaching, AQE back on
    # when it IS the topic — 10.3)
    .config("spark.sql.adaptive.enabled", "false")
    # note: shuffle.partitions deliberately NOT pinned to 8 this time —
    # we want to catch the 200 default red-handed
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

SLOTS = spark.sparkContext.defaultParallelism
print(f"master = {spark.sparkContext.master}, task slots = {SLOTS}")

## The three populations, counted

Partitions are data, tasks are work, slots are capacity. Only the
first is a property of the DataFrame — inspect it any time.

In [ ]:
df = spark.range(1_000_000)
print("partitions:", df.rdd.getNumPartitions())   # defaults to SLOTS

# tasks: run an action, then check the UI — Jobs tab, the job's stage
# shows exactly that many tasks (one per partition)
df.count()

# capacity never shows on the DataFrame — it's a cluster property
print("slots:", SLOTS, "→ at most", SLOTS, "of those tasks ran at once")

## Waves on a wall clock

Each partition below costs exactly one second of "work", so stage
duration ≈ number of waves. Watch `ceil(partitions / slots)` become
wall time — including the brutal `2×slots + 1` case, where one
straggler partition buys a whole extra wave.

In [ ]:
import time

def one_second_of_work(partition):
    time.sleep(1.0)
    yield sum(1 for _ in partition)

for n in [SLOTS, 2 * SLOTS, 2 * SLOTS + 1]:
    rdd = spark.sparkContext.parallelize(range(10_000), n)
    start = time.perf_counter()
    rdd.mapPartitions(one_second_of_work).collect()
    waves = -(-n // SLOTS)   # ceil division
    print(f"{n:>3} partitions / {SLOTS} slots = {waves} wave(s) → "
          f"{time.perf_counter() - start:4.1f} s")

## Partition-count source #1: files

A scan targets `spark.sql.files.maxPartitionBytes` (128 MB) chunks.
Our lab files are tiny, so we shrink the knob instead of the data —
same mechanics, laptop scale.

In [ ]:
# ~10 MB of parquet to scan
(spark.range(1_500_000)
 .withColumn("payload", F.sha2(col("id").cast("string"), 256))
 .write.mode("overwrite").parquet("output/scan_me"))

for max_bytes in ["128m", "4m", "1m"]:
    spark.conf.set("spark.sql.files.maxPartitionBytes", max_bytes)
    n = spark.read.parquet("output/scan_me").rdd.getNumPartitions()
    print(f"maxPartitionBytes={max_bytes:>5} → {n:>3} scan partitions")

spark.conf.set("spark.sql.files.maxPartitionBytes", "128m")  # back to default

## Partition-count source #2: shuffles

Every wide transformation re-deals into `spark.sql.shuffle.partitions`
chunks. Nobody chose 200 for this 41-row file — it's just the default
(the full story is lesson 10.3).

In [ ]:
orders = spark.read.csv(f"{DATA_DIR}/orders.csv",
                        header=True, inferSchema=True)

print("before groupBy:", orders.rdd.getNumPartitions())
print("after groupBy: ",
      orders.groupBy("country").count().rdd.getNumPartitions())

spark.conf.set("spark.sql.shuffle.partitions", SLOTS)
print("after setting shuffle.partitions =", SLOTS, "→",
      orders.groupBy("country").count().rdd.getNumPartitions())

## Both failure modes, side by side

The same aggregation over the same rows: first as `too few` fat
partitions (idle slots), then as far `too many` skinny ones
(scheduling overhead), then a sane count. After running, open the
Stages tab and compare, per stage: task count, median vs max
duration, and input size per task.

In [ ]:
big = (spark.range(20_000_000)
       .withColumn("key", (col("id") % 1000).cast("int"))
       .withColumn("payload", F.sha2(col("id").cast("string"), 256)))

def bench(label, n_partitions):
    start = time.perf_counter()
    (big.repartition(n_partitions)
        .groupBy("key").agg(F.count("*"), F.max("payload"))
        .write.format("noop").mode("overwrite").save())
    print(f"{label:>28}: {time.perf_counter() - start:5.1f} s")

bench(f"too few ({max(SLOTS // 4, 1)})", max(SLOTS // 4, 1))
bench("too many (800)", 800)
bench(f"about right ({2 * SLOTS})", 2 * SLOTS)

In [ ]:
import shutil
shutil.rmtree("output", ignore_errors=True)   # self-clean

## Exercises

1. Predict-then-verify: before running, write down the partition
   count you expect for (a) `spark.range(10)`, (b) reading
   `events.jsonl`, (c) `orders.join(orders, "order_id")` with
   `shuffle.partitions` back at 200. Then check all three.
2. Extend the wave experiment to `4×SLOTS` and `4×SLOTS + 1`
   partitions. How does the relative cost of the straggler wave
   change as total waves grow? Connect to the "2–4× slots" folk rule.
3. Gzip one copy of the parquet data as a single `.csv.gz` and read
   it. Partition count? Now find the stage in the UI and note the
   task count and duration. Which config could have helped? (Trick
   question — reread the lesson.)
4. In the bad-job demo, pull the Stages-tab numbers for the
   `too many (800)` run: median task duration vs scheduler delay.
   At what task duration does overhead stop mattering, roughly?
5. Using only `spark.conf`, make the file-scan demo produce exactly
   one partition per parquet file. Which TWO configs did you have
   to reason about, and why isn't `maxPartitionBytes` alone enough?

In [ ]:
# your exercise code here